<a href="https://colab.research.google.com/github/agmCorp/colab/blob/main/PyArg-for-dummies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install python-argumentation

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.2/170.2 kB 1.3 MB/s eta 0:00:00


In [2]:
from py_arg.aspic_classes.argumentation_system import ArgumentationSystem
from py_arg.aspic_classes.argumentation_theory import ArgumentationTheory
from py_arg.aspic_classes.defeasible_rule import DefeasibleRule
from py_arg.aspic_classes.literal import Literal
from py_arg.aspic_classes.strict_rule import StrictRule


def get_argumentation_theory():
    # Language
    literal_str_list = ['a', 'p', 'q', 'r', 's', 't']
    literal_str_list += ['-' + literal_str for literal_str in literal_str_list]
    literal_str_list += ['~' + literal_str for literal_str in literal_str_list]
    language = {literal_str: Literal(literal_str)
                for literal_str in literal_str_list}

    # Contradiction function
    contraries_and_contradictories = {literal_str: [] for literal_str in language.keys()}
    for literal_str in language.keys():
        if literal_str[0] in ('~', '-'):
            contraries_and_contradictories[literal_str].append(language[literal_str[1:]])
        else:
            contraries_and_contradictories[literal_str].append(language['-' + literal_str])

    # Strict rules
    strict_rules = [StrictRule('s1', {language['t'], language['q']}, language['-p'])]

    # Defeasible rules
    d1 = DefeasibleRule('d1', {language['~s']}, language['t'])
    d2 = DefeasibleRule('d2', {language['r']}, language['q'])
    d3 = DefeasibleRule('d3', {language['a']}, language['p'])
    defeasible_rules = [d1, d2, d3]

    # n (naming from defeasible rules to literals) and updating the contradiction function
    for defeasible_rule in defeasible_rules:
        defeasible_rule_literal = Literal.from_defeasible_rule(defeasible_rule)
        defeasible_rule_literal_negation = Literal.from_defeasible_rule_negation(defeasible_rule)
        language[str(defeasible_rule_literal)] = defeasible_rule_literal
        language[str(defeasible_rule_literal_negation)] = defeasible_rule_literal_negation
        contraries_and_contradictories[str(defeasible_rule_literal)] = [defeasible_rule_literal_negation]
        contraries_and_contradictories[str(defeasible_rule_literal_negation)] = [defeasible_rule_literal]

    # Argumentation system
    arg_sys = ArgumentationSystem(language, contraries_and_contradictories, strict_rules, defeasible_rules)

    # Knowledge base
    axioms = []
    ordinary_premises = [language[literal_str] for literal_str in ['a', 'r', '-r', '~s']]

    # Argumentation theory
    arg_theory = ArgumentationTheory(arg_sys, axioms, ordinary_premises)
    return arg_theory

In [3]:
argumentation_theory = get_argumentation_theory()

for argument in argumentation_theory.all_arguments:
    print('The argument is: ' + str(argument))
    print('Conclusion: ' + str(argument.conclusion))
    print('Premises: {' + ', '.join(str(premise) for premise in argument.premises) + '}')
    print('Strict rules: {' + ', '.join(str(rule) for rule in argument.strict_rules) + '}')
    print('Defeasible rules: {' + ', '.join(str(rule) for rule in argument.defeasible_rules) + '}')
    print('Top rule: ' + str(argument.top_rule))
    print()

The argument is: a (ordinary premise)
Conclusion: a
Premises: {a}
Strict rules: {}
Defeasible rules: {}
Top rule: None

The argument is: [a (ordinary premise)=>p]
Conclusion: p
Premises: {a}
Strict rules: {}
Defeasible rules: {a=>p}
Top rule: a=>p

The argument is: [r (ordinary premise)=>q]
Conclusion: q
Premises: {r}
Strict rules: {}
Defeasible rules: {r=>q}
Top rule: r=>q

The argument is: r (ordinary premise)
Conclusion: r
Premises: {r}
Strict rules: {}
Defeasible rules: {}
Top rule: None

The argument is: [~s (ordinary premise)=>t]
Conclusion: t
Premises: {~s}
Strict rules: {}
Defeasible rules: {~s=>t}
Top rule: ~s=>t

The argument is: [[r (ordinary premise)=>q],[~s (ordinary premise)=>t]->-p]
Conclusion: -p
Premises: {~s, r}
Strict rules: {q,t->-p}
Defeasible rules: {r=>q, ~s=>t}
Top rule: q,t->-p

The argument is: -r (ordinary premise)
Conclusion: -r
Premises: {-r}
Strict rules: {}
Defeasible rules: {}
Top rule: None

The argument is: ~s (ordinary premise)
Conclusion: ~s
Premises

In [4]:
argumentation_theory = get_argumentation_theory()

# All attacks, independent of type
for attack in argumentation_theory.all_attacks:
    print(attack)

(r (ordinary premise), -r (ordinary premise))
([[r (ordinary premise)=>q],[~s (ordinary premise)=>t]->-p], [a (ordinary premise)=>p])
(-r (ordinary premise), [r (ordinary premise)=>q])
(-r (ordinary premise), r (ordinary premise))
(-r (ordinary premise), [[r (ordinary premise)=>q],[~s (ordinary premise)=>t]->-p])


In [5]:
argumentation_theory = get_argumentation_theory()

# All undercutters
all_underminers = [(argument_a, argument_b)
    for argument_a in argumentation_theory.all_arguments
    for argument_b in argumentation_theory.all_arguments
    if argumentation_theory.undermines(argument_a, argument_b)]
print('*Underminers:*')
for attack in all_underminers:
    print(attack)

*Underminers:*
(r (ordinary premise), -r (ordinary premise))
(-r (ordinary premise), [r (ordinary premise)=>q])
(-r (ordinary premise), r (ordinary premise))
(-r (ordinary premise), [[r (ordinary premise)=>q],[~s (ordinary premise)=>t]->-p])


In [6]:
arg_theory = get_argumentation_theory()
af = arg_theory.create_abstract_argumentation_framework('af')

arg_for_r = af.get_argument('r (ordinary premise)')
defeaters_of_r = arg_for_r.get_ingoing_defeat_arguments
print('*Defeaters of the argument for r*')
for defeater in defeaters_of_r:
    print(defeater)
print()

arg_for_not_r = af.get_argument('-r (ordinary premise)')
defeated_by_not_r = arg_for_not_r.get_outgoing_defeat_arguments
print('*Arguments defeated by the argument for not r*')
for defeated in defeated_by_not_r:
    print(defeated)
print()

*Defeaters of the argument for r*
-r (ordinary premise)

*Arguments defeated by the argument for not r*
[r (ordinary premise)=>q]
r (ordinary premise)
[[r (ordinary premise)=>q],[~s (ordinary premise)=>t]->-p]



In [7]:
from py_arg.algorithms.semantics.get_complete_extensions import get_complete_extensions
from py_arg.algorithms.semantics.get_grounded_extension import get_grounded_extension


grounded_extension = get_grounded_extension(af)
print('*Grounded extension:*')
print('{' + ', '.join(str(grounded) for grounded in grounded_extension) + '}')
print()

complete_extensions = get_complete_extensions(af)
print('*Complete extensions:*')
for complete_extension in complete_extensions:
    print('{' + ', '.join(str(complete) for complete in complete_extension) + '}')
print()

*Grounded extension:*
{~s (ordinary premise), [~s (ordinary premise)=>t], a (ordinary premise)}

*Complete extensions:*
{~s (ordinary premise), [[r (ordinary premise)=>q],[~s (ordinary premise)=>t]->-p], [~s (ordinary premise)=>t], a (ordinary premise), r (ordinary premise), [r (ordinary premise)=>q]}
{~s (ordinary premise), -r (ordinary premise), [~s (ordinary premise)=>t], [a (ordinary premise)=>p], a (ordinary premise)}
{~s (ordinary premise), [~s (ordinary premise)=>t], a (ordinary premise)}



In [8]:
from py_arg.aba_classes.rule import Rule
from py_arg.aba_classes.aba_framework import ABAF
from py_arg.aba_classes.semantics import get_preferred_extensions


language = {'happy',
            'eating',
            'good_food',
            'not_eating',
            'no_fork',
            'dirty_hands',
            'fork',
            'clean_hands'}
rules = {Rule('Rule1', {'good_food', 'eating'}, 'happy'),
         Rule('Rule2', set(), 'good_food'),
         Rule('Rule3', {'no_fork', 'dirty_hands'}, 'not_eating')}
assumptions = {'eating', 'no_fork', 'dirty_hands'}
contraries = {'eating': 'not_eating',
              'no_fork': 'fork',
              'dirty_hands': 'clean_hands'}

# Construct the framework
aba_framework = ABAF(assumptions, rules, language, contraries)

# Get preferred extensions
extensions = get_preferred_extensions.get_preferred_extensions(aba_framework)
print(extensions)

{frozenset({'no_fork', 'dirty_hands'})}
